# Día 1 · Planteamiento del problema, población y muestra

**Apartados del documento que cubre este día:** 1 (Planteamiento del problema) y 2 (Identificación de población y muestra).

**Al terminar este día, cada integrante del equipo debe poder explicar sin leer:**
1. Qué problema estudiamos y por qué importa.
2. Qué es la población y qué es la muestra en nuestro proyecto, con sus números.
3. De dónde sale el número 1,000 (el tamaño de la muestra).
4. Cómo se eligieron los 1,000 clientes y por qué así.
5. Cómo comprobamos que la muestra se parece a la población.

Al final hay una lista de **preguntas que puede hacer el profesor**, con su respuesta.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

---
## Paso 1 · Qué pide el documento

**Apartado 1 · Planteamiento del problema.** El documento pide responder cuatro preguntas:
- ¿Qué problema están investigando?
- ¿Por qué es relevante?
- ¿Qué quieren conocer?
- ¿Cuál es la población de interés?

**Apartado 2 · Identificación de población y muestra.** El documento dice que este apartado "es fundamental" y pide:
- Población y muestra.
- Tamaño de la muestra.
- Método de selección.
- Criterios de inclusión y de exclusión.
- Por qué la muestra puede representar razonablemente a la población.

Los siguientes pasos responden cada punto con lo que hicimos en el proyecto.

---
## Paso 2 · Las cuatro respuestas del planteamiento

| Pregunta del documento | Respuesta del proyecto |
|---|---|
| ¿Qué problema investigamos? | La **deserción de clientes** de un banco: clientes que cierran su cuenta y se van. En inglés se le llama *churn*. |
| ¿Por qué es relevante? | Cada cliente que se va se lleva su dinero depositado y las comisiones que pagaba. Conseguir un cliente nuevo cuesta más que conservar uno. |
| ¿Qué queremos conocer? | Cuatro números de toda la cartera de clientes (ver tabla siguiente). |
| ¿Cuál es la población? | Los **10,000 clientes** del banco en Francia, Alemania y España. |

Los cuatro números que queremos conocer:

| Pregunta | Qué número buscamos | Tipo de número |
|---|---|---|
| P1 | Qué porcentaje de clientes abandona el banco | Una proporción |
| P2 | Cuánto dinero tiene en promedio un cliente que abandona | Una media |
| P3 | Cuántos puntos porcentuales más abandonan los clientes inactivos que los activos | Una resta de dos proporciones |
| P4 | Cuántos años más tienen, en promedio, los que abandonan que los que se quedan | Una resta de dos medias |

**Por qué cuatro preguntas:** el documento exige "al menos dos técnicas de estimación" y da como opciones media, diferencia de medias, proporción y diferencia de proporciones. Usamos las cuatro.

Primero cargamos los datos:

In [2]:
RAIZ = Path.cwd().parent if Path.cwd().name == "clases" else Path.cwd()
poblacion = pd.read_csv(RAIZ / "data" / "processed" / "poblacion_clientes.csv")
muestra = pd.read_csv(RAIZ / "data" / "processed" / "muestra_clientes_n1000.csv")
print("Clientes en la población:", len(poblacion))
print("Clientes en la muestra:", len(muestra))

Clientes en la población: 10000
Clientes en la muestra: 1000


**Comprobación:** la población tiene 10,000 clientes y la muestra 1,000.

---
## Paso 3 · Población y muestra

- **Población:** el grupo completo sobre el que queremos sacar conclusiones. Su tamaño se escribe con **N mayúscula**. Aquí, N = 10,000.
- **Muestra:** la parte de la población que realmente analizamos. Su tamaño se escribe con **n minúscula**. Aquí, n = 1,000.

**¿Por qué no analizar a los 10,000 si los tenemos?** Esta es la pregunta más probable del profesor. Hay dos razones:

1. **En un banco real, confirmar el abandono cuesta.** El sistema marca muchos cierres como "inactividad"; para saber si un cliente realmente se fue, un analista revisa su expediente uno por uno. Revisar 1,000 expedientes cuesta la décima parte que revisar 10,000.
2. **El documento del proyecto pide trabajar con una muestra** y sacar conclusiones sobre la población. Que tengamos los 10,000 nos da una ventaja: al final podemos **comprobar** si nuestras conclusiones acertaron.

**Comprobación:** la muestra es el 10 % de la población.

In [3]:
N = len(poblacion)
n = len(muestra)
fraccion_muestreo = n / N
print("N (población):", N)
print("n (muestra):", n)
print("Fracción de la población que está en la muestra:", fraccion_muestreo)

N (población): 10000
n (muestra): 1000
Fracción de la población que está en la muestra: 0.1


---
## Paso 4 · Parámetro y estadístico

Estos dos términos aparecen en el apartado 4 del documento, pero conviene entenderlos desde hoy.

- **Parámetro:** un número calculado con **toda la población**. Normalmente no se conoce. Se escribe sin marca: $p$ para una proporción, $\mu$ (se lee "miu") para una media.
- **Estadístico:** el mismo número, pero calculado **con la muestra**. Se escribe con una marca encima: $\hat p$ (se lee "p gorro"), $\bar x$ (se lee "x barra").

El estadístico se usa para estimar el parámetro. En nuestro proyecto podemos calcular los dos y compararlos:

In [4]:
p_poblacion = poblacion["abandono"].mean()
p_muestra = muestra["abandono"].mean()
diferencia = p_muestra - p_poblacion
print("Parámetro p (proporción de abandono en los 10,000):", round(p_poblacion, 4))
print("Estadístico p gorro (proporción de abandono en los 1,000):", round(p_muestra, 4))
print("Diferencia entre los dos:", round(diferencia, 4))

Parámetro p (proporción de abandono en los 10,000): 0.2037
Estadístico p gorro (proporción de abandono en los 1,000): 0.206
Diferencia entre los dos: 0.0023


**Lectura:** la muestra dio 20.6 % y el valor real es 20.37 %. La diferencia es de 0.23 puntos porcentuales. A esa diferencia se le llama **error de muestreo**: aparece porque analizamos una parte y no el total. En la vida real no se conoce el parámetro, así que no se puede calcular esta diferencia; por eso, en los días 3 y 4, aprenderemos a dar un **rango** en lugar de un solo número.

---
## Paso 5 · El tamaño de la muestra: de dónde sale 1,000

Para elegir cuántos clientes revisar se usa la fórmula del tamaño de muestra para una proporción:

$$n_0 = \frac{z^2 \cdot p \cdot (1 - p)}{E^2}$$

Cada símbolo es una decisión del equipo:

| Símbolo | Qué es | Valor que elegimos | Por qué |
|---|---|---|---|
| $E$ | El error máximo que aceptamos, hacia arriba o hacia abajo | 0.03 (3 puntos porcentuales) | Para decidir una campaña basta saber el abandono con ±3 puntos |
| $z$ | Un número que depende de qué tan seguido queremos acertar | 1.96 | Corresponde a acertar el 95 % de las veces (se explica en el día 3) |
| $p$ | La proporción que esperamos encontrar | 0.5 | Antes del estudio no sabíamos la tasa de abandono. Con 0.5 la fórmula da el tamaño más grande posible, así que nunca nos quedamos cortos |

**¿Por qué 0.5 da el tamaño más grande?** El término $p \cdot (1-p)$ vale como máximo 0.25, y ese máximo ocurre en $p = 0.5$. Se comprueba con tres valores:

In [5]:
producto_con_010 = 0.10 * (1 - 0.10)
producto_con_030 = 0.30 * (1 - 0.30)
producto_con_050 = 0.50 * (1 - 0.50)
print("p(1-p) con p = 0.10:", round(producto_con_010, 2))
print("p(1-p) con p = 0.30:", producto_con_030)
print("p(1-p) con p = 0.50:", producto_con_050)

p(1-p) con p = 0.10: 0.09
p(1-p) con p = 0.30: 0.21
p(1-p) con p = 0.50: 0.25


Ahora la fórmula, una operación por línea:

$$n_0 = \frac{1.96^2 \cdot 0.5 \cdot 0.5}{0.03^2} = \frac{3.8416 \cdot 0.25}{0.0009} = \frac{0.9604}{0.0009} = 1067.1$$

In [6]:
z = 1.96
p_esperada = 0.5
error_maximo = 0.03
z_al_cuadrado = z ** 2
producto_p = p_esperada * (1 - p_esperada)
numerador = z_al_cuadrado * producto_p
denominador = error_maximo ** 2
n0 = numerador / denominador
print("z al cuadrado:", round(z_al_cuadrado, 4))
print("p(1-p):", producto_p)
print("Numerador:", round(numerador, 4))
print("Denominador (E al cuadrado):", round(denominador, 4))
print("n0 (tamaño sin corregir):", round(n0, 1))

z al cuadrado: 3.8416
p(1-p): 0.25
Numerador: 0.9604
Denominador (E al cuadrado): 0.0009
n0 (tamaño sin corregir): 1067.1


### Corrección porque la población es de sólo 10,000

La fórmula anterior supone una población muy grande. Como la nuestra tiene 10,000 clientes, se corrige:

$$n = \frac{n_0}{1 + \dfrac{n_0 - 1}{N}} = \frac{1067.1}{1 + \dfrac{1066.1}{10000}} = \frac{1067.1}{1.1066} = 964.3$$

La corrección **reduce** el número de clientes necesarios. Razón: cuando la muestra es una parte grande del total, cada cliente revisado aporta más información sobre ese total.

In [7]:
cociente = (n0 - 1) / N
divisor = 1 + cociente
n_corregido = n0 / divisor
print("(n0 - 1) / N:", round(cociente, 4))
print("Divisor:", round(divisor, 4))
print("n corregido:", round(n_corregido, 1))

(n0 - 1) / N: 0.1066
Divisor: 1.1066
n corregido: 964.3


**Decisión:** la fórmula pide al menos 965 clientes. Elegimos **1,000**: es un número redondo y deja un pequeño margen para las preguntas P3 y P4, que dividen la muestra en dos grupos.

**Comprobación:** 1,000 es mayor o igual que el mínimo calculado.

In [8]:
print("¿1,000 cumple el mínimo?:", 1000 >= np.ceil(n_corregido))

¿1,000 cumple el mínimo?: True


---
## Paso 6 · Método de selección: muestreo aleatorio simple

Elegimos a los 1,000 clientes con **muestreo aleatorio simple sin reemplazo**:
- **Aleatorio:** la computadora elige a los clientes por sorteo; ninguna persona decide quién entra.
- **Simple:** todos los clientes tienen la misma probabilidad de salir: 1,000 / 10,000 = 10 %.
- **Sin reemplazo:** un cliente no puede salir dos veces.

**Por qué este método:**
1. Evita el **sesgo**: si una persona elige a quién revisar (por ejemplo, sólo clientes de Francia), la muestra puede quedar inclinada hacia un tipo de cliente.
2. Las fórmulas del curso (las de los días 3 y 4) **suponen** que la muestra se eligió así.
3. Se puede hacer porque tenemos la lista completa de los 10,000 clientes.

**La semilla.** La computadora sortea a partir de un número inicial llamado **semilla**. Con la misma semilla, el sorteo da siempre la misma muestra; con otra semilla, da otra muestra. Usamos la semilla **2026** para que el profesor pueda repetir el sorteo y obtener exactamente nuestros 1,000 clientes.

In [9]:
generador_2026 = np.random.default_rng(2026)
posiciones_2026 = generador_2026.choice(N, size=n, replace=False)
muestra_repetida = poblacion.iloc[posiciones_2026]

generador_otra_semilla = np.random.default_rng(7)
posiciones_otra = generador_otra_semilla.choice(N, size=n, replace=False)
otra_muestra = poblacion.iloc[posiciones_otra]

print("Primeros 3 clientes con semilla 2026:", muestra_repetida["id_cliente"].head(3).tolist())
print("Primeros 3 clientes de la muestra guardada:", muestra["id_cliente"].head(3).tolist())
print("Primeros 3 clientes con semilla 7:", otra_muestra["id_cliente"].head(3).tolist())

Primeros 3 clientes con semilla 2026: [15675256, 15653780, 15711572]
Primeros 3 clientes de la muestra guardada: [15675256, 15653780, 15711572]
Primeros 3 clientes con semilla 7: [15604813, 15708608, 15689152]


**Comprobación:** con la semilla 2026 salen los mismos clientes que en la muestra guardada; con la semilla 7 salen otros. Además, no hay clientes repetidos:

In [10]:
print("¿Hay clientes repetidos en la muestra?:", muestra["id_cliente"].duplicated().any())

¿Hay clientes repetidos en la muestra?: False


---
## Paso 7 · Criterios de inclusión y exclusión

- **Criterios de inclusión:** las condiciones que debe cumplir un cliente para formar parte del estudio.
- **Criterios de exclusión:** las condiciones que lo dejan fuera.

| Inclusión | Por qué |
|---|---|
| Tener 18 años o más | El banco sólo atiende a mayores de edad |
| Tener registrado si abandonó o no | Sin ese dato no aporta a ninguna pregunta |
| Ser cliente de Francia, Alemania o España | Son los países del banco |

| Exclusión | Por qué |
|---|---|
| Registros duplicados | Un cliente repetido tendría el doble de probabilidad de salir en el sorteo |
| Datos imposibles (edad de 150 años, saldo negativo) | Serían errores de captura |
| Datos personales que no se usan (apellido) | La ley europea de protección de datos pide guardar sólo lo necesario |

**Comprobación:** todos los clientes cumplen la inclusión, por eso la población es de 10,000.

In [11]:
mayores_de_edad = (poblacion["edad"] >= 18).all()
abandono_registrado = poblacion["abandono"].notna().all()
sin_duplicados = not poblacion["id_cliente"].duplicated().any()
print("¿Todos tienen 18 años o más?:", mayores_de_edad)
print("¿Todos tienen el abandono registrado?:", abandono_registrado)
print("¿No hay clientes duplicados?:", sin_duplicados)

¿Todos tienen 18 años o más?: True
¿Todos tienen el abandono registrado?: True
¿No hay clientes duplicados?: True


---
## Paso 8 · ¿La muestra representa a la población?

Si el sorteo salió bien, la muestra debe tener **casi los mismos porcentajes** que la población en variables que no vamos a estimar. Primero se comparan los porcentajes por país:

In [12]:
porcentaje_pais_poblacion = poblacion["pais"].value_counts(normalize=True) * 100
porcentaje_pais_muestra = muestra["pais"].value_counts(normalize=True) * 100
print("Porcentaje por país en la población:")
print(porcentaje_pais_poblacion.round(1))
print("Porcentaje por país en la muestra:")
print(porcentaje_pais_muestra.round(1))

Porcentaje por país en la población:
pais
Francia     50.1
Alemania    25.1
España      24.8
Name: proportion, dtype: float64
Porcentaje por país en la muestra:


pais
Francia     49.7
Alemania    27.6
España      22.7
Name: proportion, dtype: float64


Las diferencias van de 0.4 a 2.5 puntos porcentuales. ¿Son normales en un sorteo o son demasiado grandes? Para responder se usa la **prueba ji cuadrada de bondad de ajuste** (se escribe χ²).

**Hipótesis de la prueba:**
- **H0 (hipótesis nula):** la distribución por país de la muestra es igual a la de la población. Las diferencias se deben sólo al sorteo.
- **H1 (hipótesis alternativa):** la distribución por país de la muestra es distinta a la de la población.
- **Nivel de significancia:** α = 0.05.

**Regla de decisión:** si el p-valor es menor que 0.05, se rechaza H0. Si es mayor o igual, no se rechaza H0.

**Qué es el p-valor:** la probabilidad de obtener diferencias tan grandes como las observadas **si H0 fuera cierta**. Un p-valor grande indica que las diferencias son comunes en un sorteo.

**Supuestos de la prueba:**
1. Los datos son conteos de una muestra aleatoria. Se cumple (muestreo aleatorio simple).
2. Cada cliente cuenta en una sola categoría. Se cumple (cada cliente tiene un solo país).
3. El conteo esperado en cada categoría es de al menos 5. Se comprueba abajo.

In [13]:
conteo_observado = muestra["pais"].value_counts().sort_index()
proporcion_poblacion = poblacion["pais"].value_counts(normalize=True).sort_index()
conteo_esperado = proporcion_poblacion * n
print("Conteo observado en la muestra:")
print(conteo_observado)
print("Conteo esperado si la muestra fuera igual a la población:")
print(conteo_esperado.round(1))
print("¿Todos los conteos esperados son de al menos 5?:", (conteo_esperado >= 5).all())

Conteo observado en la muestra:
pais
Alemania    276
España      227
Francia     497
Name: count, dtype: int64
Conteo esperado si la muestra fuera igual a la población:
pais
Alemania    250.9
España      247.7
Francia     501.4
Name: proportion, dtype: float64
¿Todos los conteos esperados son de al menos 5?: True


In [14]:
resultado_chi2 = stats.chisquare(conteo_observado, conteo_esperado)
estadistico_chi2 = resultado_chi2.statistic
p_valor = resultado_chi2.pvalue
print("Estadístico ji cuadrada:", round(estadistico_chi2, 3))
print("p-valor:", round(p_valor, 3))

Estadístico ji cuadrada: 4.279
p-valor: 0.118


**Conclusión:** No se rechaza H0 (p = 0.118 ≥ 0.05): no hay evidencia de que la distribución por país de la muestra sea distinta de la de la población.

**Importancia práctica:** las diferencias observadas van de 0.4 a 2.5 puntos porcentuales por país. No cambian ninguna conclusión del proyecto.

En el proyecto se hizo la misma prueba con género, actividad, número de productos y grupo de edad. Todos los p-valores fueron mayores a 0.05 (de 0.12 a 0.92). Por eso decimos que la muestra **representa razonablemente** a la población.

**Cuidado con la redacción:** "no se rechaza H0" no significa "se demostró que son iguales". Significa que los datos no muestran una diferencia mayor que la que produce el sorteo.

---
## Preguntas que puede hacer el profesor (apartados 1 y 2)

**1. ¿Por qué tomaron una muestra si tenían a los 10,000 clientes?**
Porque en un banco real confirmar el abandono exige revisar cada expediente, y eso cuesta. Además, el proyecto pide estimar a partir de una muestra. Tener a los 10,000 nos permitió comprobar al final si acertamos.

**2. ¿De dónde sale el tamaño de 1,000?**
De la fórmula del tamaño de muestra para una proporción, con error máximo de ±3 puntos, 95 % de confianza y p = 0.5. Da 1,067; con la corrección por población de 10,000 da 965. Redondeamos a 1,000.

**3. ¿Por qué usaron p = 0.5 si el abandono real es cercano a 0.20?**
Porque antes del estudio no sabíamos la tasa de abandono. Con 0.5 la fórmula da el tamaño más grande posible, así que la muestra alcanza para cualquier valor real.

**4. ¿Por qué ±3 puntos y no ±1?**
Para decidir una campaña de retención basta con ±3 puntos. Con ±1 punto la muestra tendría que ser de unos 4,900 clientes, casi la mitad de la población.

**5. ¿Cómo eligieron a los clientes?**
Por muestreo aleatorio simple sin reemplazo: la computadora sorteó 1,000 clientes, todos con la misma probabilidad, usando la semilla 2026.

**6. ¿Qué es la semilla y para qué sirve?**
Es el número inicial del sorteo de la computadora. Con la misma semilla se obtiene la misma muestra, así cualquiera puede repetir nuestro trabajo.

**7. ¿Cuáles fueron los criterios de inclusión y exclusión?**
Inclusión: mayores de edad, con abandono registrado y de los tres países. Exclusión: duplicados, datos imposibles y datos personales innecesarios.

**8. ¿Por qué su muestra representa a la población?**
Por el método (sorteo con la misma probabilidad para todos) y porque lo comprobamos: los porcentajes por país, género, actividad, productos y edad son casi iguales, y la prueba ji cuadrada no encontró diferencias (todos los p-valores mayores a 0.05).

**9. ¿Qué diferencia hay entre parámetro y estadístico?**
El parámetro es el número de toda la población (desconocido en la práctica); el estadístico es el mismo número calculado con la muestra y se usa para estimarlo.

**10. ¿Qué limitación tiene su población?**
El dataset no identifica al banco ni las fechas. Por eso las conclusiones aplican a estos 10,000 clientes, no a todos los bancos de Europa.

---
## Tarea del Día 1 (para todo el equipo)

Cada integrante explica en voz alta, **sin leer**, las respuestas a las preguntas 1, 2, 5 y 8. Si alguien se traba, vuelvan al paso correspondiente.

➡️ **Día 2:** Apartado 3 · Estadística descriptiva y las gráficas.